# Research, read the answer, execute the saved study

Run the cells from top to bottom using **Epoch SDK (local prototype)**.

We supply two synthetic observations: **2024 = 10 USD**, **2025 = 15 USD**.
The expected answer has a two-row table, JSON with `row_count=2` and `total_sales=25`,
and a short text explanation. These are software demonstration values.

The research cell starts real EpochAI/model and Backend work. A saved-study execution
is a separate action. Re-running the same operation ID and inputs attaches to the
same work. Choose a new ID when you want a new task or different parameters.

## 1. Connect to the running services

The prepared notebook server supplies your Epoch identity and both service URLs. Credentials are not stored in this notebook.

The functions imported from `notebook_support.py` are walkthrough helpers.
`connect()` calls the SDK's `EpochClient.connect(...)` with your Epoch login and
both service addresses, then keeps the resulting client in the variable `epoch`.
`check_services()` prints the installed SDK versions and reads the services' health
endpoints. `await` means this cell waits for those network checks to finish.
This setup step prepares access; the later `epoch.research(...)` call starts research.


In [ ]:
# Import the SDK types and the small helpers used by this walkthrough.
import json
from pathlib import Path
from epoch_sdk import RunRequest
from IPython.display import JSON, display
from stratifyx import DatePeriodConfig
from epoch_sdk.research import ResearchCompleted
from epoch_sdk.supervisor import NeedsInput
from notebook_support import (
    connect, check_services, operation_name,
    wait_research, wait_execution, show_answer, check_demo_values,
)

# Create the SDK client using the Epoch login and service URLs supplied
# by the notebook server. Later cells use this object as "epoch".
epoch = connect()
# Print the installed SDK versions and check that Backend and EpochAI respond.
# "await" waits for those network checks to finish. Research starts in step 3.
await check_services()

## 2. Describe the task and requested answer

The input file is [cases/annual_sales.json](cases/annual_sales.json), in the **cases**
folder beside this notebook. Open that folder in the left file browser to inspect
or edit the task, supplied data, date range and requested table/JSON/text fields.

The cell below reads that local file and validates it as an SDK `RunRequest`.
Choose new operation IDs when changing accepted inputs.

In [ ]:
# Read the input file from the cases folder beside this notebook.
# The research and the later saved-study execution have separate readable IDs.
# Expand the displayed JSON to inspect the task and output declarations.
CASE_FILE = Path("cases/annual_sales.json")
case_data = json.loads(CASE_FILE.read_text())
request = RunRequest.model_validate(case_data)
print("Input case:", CASE_FILE.resolve())
RESEARCH_ID = operation_name("research")
EXECUTION_ID = operation_name("execute")
print("Research ID:", RESEARCH_ID)
print("Execution ID:", EXECUTION_ID)
display(JSON(request.to_wire(), expanded=False))

## 3. Research: build, execute, inspect and publish

This may take a minute. The returned answer is saved by Backend. If the model asks a question, inspect it before continuing; notebook 02 demonstrates answering and resuming.

In [ ]:
# Send the task to Epoch, or attach to it if this research ID already exists.
# If it is still running, wait for a saved answer or a clarification question.
# A question must be answered before the following result-reading cells can run.
result = await epoch.research(request, operation_id=RESEARCH_ID)
if not isinstance(result, (ResearchCompleted, NeedsInput)):
    result = await wait_research(epoch, RESEARCH_ID)
if isinstance(result, NeedsInput):
    print(result.feedback())
else:
    print("Research complete:", result.response.response_id)
    print("Reusable study:", result.reusable_study.study_id if result.reusable_study else None)

## 4. Read the published table, JSON and text

The table comes from saved computed data. Displaying it does not request another dashboard or start another study.

In [ ]:
# An assert stops this cell if research has not produced a saved answer yet.
# show_answer() displays the saved answer and fetches the table rows for viewing.
assert isinstance(result, ResearchCompleted), "Answer the pending question before reading a completed answer."
await show_answer(epoch, result)

## 5. Verify all rows, including pagination

A page size of one deliberately exercises both pages. The total table is not limited to a single transfer.

In [ ]:
# Read this small table one page at a time and collect its rows.
# page_size=1 is deliberately small so you can see pagination working.
# check_demo_values() then checks the two known rows and the JSON total of 25.
rows = []
async for page in epoch.iter_response_table(
    result.response.response_id, "annual_sales",
    version=result.response.version, page_size=1,
):
    rows.extend(page.rows)
    print("Page offset:", page.offset, "next:", page.next_offset)
check_demo_values(result, rows)

## 6. Execute the saved study directly through Backend

This reuses the prepared study and creates one new execution for this execution operation ID. It does not repeat discovery or ask the supervisor to rebuild the study.

In [ ]:
# Use the saved study returned by research and specify the execution date range.
# execute_saved() starts or attaches to one Backend job under EXECUTION_ID.
# wait_execution() reads that job until it completes; it does not rebuild the study.
# The assert below checks that the completed job used the same saved study.
assert result.reusable_study is not None
period = DatePeriodConfig(start="2024-01-01", end="2025-12-31")
execution = await epoch.execute_saved(
    result.reusable_study,
    period=period,
    operation_id=EXECUTION_ID,
)
execution = await wait_execution(epoch, EXECUTION_ID)
assert execution.execution.definition_id == result.reusable_study.study_id
print("Study:", execution.execution.definition_id)
print("New execution:", execution.execution.id)
print("Status:", execution.execution.status.value)

## 7. Repeat the call: attach to the same execution

Run this cell repeatedly. A new intended execution needs a new operation ID.

In [ ]:
# Repeat exactly the same study, dates and execution ID.
# The returned execution ID should match the job from the preceding cell.
same_execution = await epoch.execute_saved(
    result.reusable_study,
    period=period,
    operation_id=EXECUTION_ID,
)
assert same_execution.execution.id == execution.execution.id
print("Same execution:", same_execution.execution.id)

## 8. The research answer keeps its original source

Prepared execution returns an execution. The research answer remains pinned to its original run and revision. A new interpretation of new results is a separate answer-producing step.

In [ ]:
# Read the answer belonging to the original research operation.
# These checks confirm that its saved answer and revision have stayed the same.
# The prepared execution has a different run ID and does not replace that answer.
original = await epoch.research_status(operation_id=RESEARCH_ID)
assert isinstance(original, ResearchCompleted)
assert original.response.response_id == result.response.response_id
assert original.response.version == result.response.version
assert original.response.primary_execution_id != execution.execution.id
print("Original answer execution:", original.response.primary_execution_id)
print("Prepared execution:", execution.execution.id)
print("Pinned answer revision:", original.response.version)

## 9. Close this client

The services keep running. Notebook 02 exercises reconnecting from a fresh client and answering clarification.

In [ ]:
# Close this Python client and its connections.
# The services and the saved studies, executions and answers remain available.
await epoch.aclose()